In [0]:
spark


In [0]:
myRange = spark.range(1000).toDF("number")
myRange.show()


In [0]:
divisBy2 = myRange.where("number % 2 = 0")
divisBy2.show()

In [0]:
divisBy2.count()

In [0]:
flightData2015 = spark.table("workspace.default.2010_summary")
flightData2015.show(5)

In [0]:
flightData2014 = spark.read.option("inferSchema", "true").option("header", "true").csv("/Volumes/workspace/default/flight_data/2014-summary.csv")
flightData2014.show(5)

In [0]:
flightData2014.take(3)

In [0]:
flightData2014.sort("count").explain()

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "5")
flightData2014.sort("count").take(2)

In [0]:
flightData2014.createOrReplaceTempView("flightData2014")

In [0]:
flightData2015 = spark.read.option("inferSchema", "true").option("header", "true").csv("/Volumes/workspace/default/flight_data/2015-summary.csv")
flightData2015.show(5)

In [0]:
flightData2015.createOrReplaceTempView("flightData2015")
sqlWay = spark.sql(""" SELECT DEST_COUNTRY_NAME, count(1) FROM flightData2015 GROUP BY DEST_COUNTRY_NAME""")

In [0]:
dataFrameWay = flightData2015.groupBy("DEST_COUNTRY_NAME").count()

In [0]:
sqlWay.explain()

In [0]:
dataFrameWay.explain()

In [0]:
spark.sql("SELECT max(count) from flightData2015").take(1)

In [0]:
from pyspark.sql.functions import max

flightData2015.select(max("count")).take(1)

In [0]:
maxSql = spark.sql("""
    SELECT DEST_COUNTRY_NAME, sum(count) as destination_total
    FROM flightData2015
    GROUP BY DEST_COUNTRY_NAME
    ORDER BY sum(count) DESC
    LIMIT 5        """)

maxSql.show()

In [0]:
from pyspark.sql.functions import desc

flightData2015.groupBy("DEST_COUNTRY_NAME").sum("count").withColumnRenamed("sum(count)", "destination_total").sort(desc("destination_total")).limit(5).show()

In [0]:
df = spark.range(500).toDF("number")
df.select(df["number"] + 10).show(10)

In [0]:
df = spark.read.option("inferSchema", "true").option("header", "true").csv("/Volumes/workspace/default/flight_data/2015-summary.csv")
df.show(5)

In [0]:
df.printSchema()

In [0]:
spark.read.format("json").load("/Volumes/workspace/default/flight_data/json/2015-summary.json").printSchema()

In [0]:
df = spark.read.option("inferSchema", "true").option("header", "true").csv("/Volumes/workspace/default/flight_data/2015-summary.csv")
df.first()

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/default/flight_data/retail_data/2010-12-01.csv")

In [0]:
df.columns

In [0]:
df.createOrReplaceTempView("dfTable")

In [0]:
df.sample(withReplacement=False, fraction=0.3, seed=5)

In [0]:
df.show(5)

In [0]:
df.Description

In [0]:
from pyspark.sql.functions import col, instr
priceFilter = col("UnitPrice") > 600
descripFilter = instr(df.Description, "POSTAGE") >= 1
df.where(df.StockCode.isin("DOT")).where(priceFilter | descripFilter).show()

In [0]:
df.show(5, False)

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/default/flight_data/retail_data/2010-12-01.csv")

In [0]:
from pyspark.sql.functions import regexp_replace, col
regex_string = "BLACK|WHITE|RED|GREEN|BLUE"
df.select(regexp_replace(col("Description"), regex_string, "COLORS").alias("color_clean"), col("Description")).show(2, False)

In [0]:
from pyspark.sql.functions import translate
df.select(translate(col("Description"), "LEET", "1337").alias("text"), col("Description")).show(2, False)

In [0]:
from pyspark.sql.functions import current_date, current_timestamp
dateDF = spark.range(10).withColumn("today", current_date()).withColumn("now", current_timestamp())
dateDF.createOrReplaceTempView("dateTable")
dateDF.printSchema()

In [0]:
from pyspark.sql.functions import date_add, date_sub, col
dateDF.select(date_sub(col("today"), 5), date_add(col("today"), 5)).show(1)

In [0]:
from pyspark.sql.functions import to_date, lit, col
dateFormat = "yyyy-dd-MM"
cleanDateDF = spark.range(1).select(
    to_date(lit("2017-12-11"), dateFormat).alias("date"),
    to_date(lit("2017-20-12"), dateFormat).alias("date2")).show()


In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/default/flight_data/retail_data/2010-12-01.csv")

In [0]:
from pyspark.sql.functions import struct, col
complexDF = df.select(struct("Description", "InvoiceNo").alias("complex"))

In [0]:
complexDF.select("complex.Description")

In [0]:
complexDF.select(col("complex").getField("Description")).show(2, False)

In [0]:
from pyspark.sql.functions import split, col
df.select(split(col("Description"), " ")).show(2, False)

In [0]:
df.select(split(col("Description"), " ").alias("array_col")).selectExpr("array_col[0]").show()

In [0]:
from pyspark.sql.functions import split, col, array_contains
df.select(array_contains(split(col("Description"), " "), "WHITE")).show(2, False)

In [0]:
from pyspark.sql.functions import split, col, explode
df.withColumn("splitted", split(col("Description"), " ")).withColumn("exploded", explode(col("splitted"))).select("Description", "InvoiceNo", "exploded").show(2)

In [0]:
from pyspark.sql.functions import split, col, explode, create_map
df.select(create_map(col("Description"), col("InvoiceNo")).alias("complex_map")).selectExpr("explode(complex_map)").show(5, False)

In [0]:
jsonDF = spark.range(1).selectExpr("""
                                   '{"myJSONKey" : {"myJSONValue" : [1,2,3]}}' as jsonString""")

In [0]:
from pyspark.sql.functions import col, get_json_object
jsonDF.select(get_json_object(col("jsonString"), "$.myJSONKey.myJSONValue[0]").alias("column")).show(2)

In [0]:
from pyspark.sql.functions import col, get_json_object, json_tuple
jsonDF.select(json_tuple(col("jsonString"), "myJSONKey")).show(2, False)